# Assignment 09: End-to-End ML Pipeline (100 points)

**Unit**: ML1 Supervised Learning (AI 300)  
**Topics**: Data preprocessing, feature engineering, model comparison, cross-validation, evaluation

---

## Background

A complete ML pipeline involves: raw data $\to$ preprocessing $\to$ feature engineering $\to$ model training $\to$ evaluation. This assignment ties together everything from Assignments 01--08.

You will use **your own implementations** from previous assignments (copy key functions as needed). The pipeline includes: standardization, polynomial features, multiple model types, cross-validation for model selection, and proper test-set evaluation with confidence intervals.

### Notation

| Symbol | Description |
|--------|-------------|
| $X_{\text{train}}$ | Training features after preprocessing |
| $X_{\text{test}}$ | Held-out test features (same preprocessing as train) |
| CV | Cross-validation |

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

> **WARNING !!!**
>
> - Beyond importing libraries/modules/classes/functions in the preceding cell, you are **NOT allowed to import anything else for the following purposes**:
>     - **As a part of your final solution.**
>     - **Temporarily import something to assist you to get a solution.**
>
>     **Rule of thumb:** Each part has its particular purpose to intentionally test you something. Do not attempt to find a shortcut to circumvent the rule.

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
Dataset: realistic binary classification with nonlinear boundary.
"""
def generate_dataset(n=500, d=10, noise=0.3, seed=42):
    rng = np.random.RandomState(seed)
    X = rng.randn(n, d)
    # Different scales per feature
    X[:, 0] *= 100
    X[:, 1] *= 0.01
    X[:, 2] *= 10
    # Nonlinear decision: depends on features 0, 1, 2, 3
    logits = (
        0.5 * (X[:, 0] / 100) +
        2.0 * (X[:, 1] / 0.01) +
        -1.0 * (X[:, 2] / 10) +
        0.3 * X[:, 3] +
        0.5 * (X[:, 0] / 100) * (X[:, 1] / 0.01) +  # interaction term
        noise * rng.randn(n)
    )
    y = (logits > 0).astype(int)
    return X, y

X_all, y_all = generate_dataset(n=500)                           # (500, 10)

# Train-test split (80-20)
idx = np.random.permutation(len(y_all))
n_train = int(0.8 * len(y_all))
X_train, y_train = X_all[idx[:n_train]], y_all[idx[:n_train]]   # (400, 10), (400,)
X_test, y_test = X_all[idx[n_train:]], y_all[idx[n_train:]]     # (100, 10), (100,)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Class balance (train): {y_train.mean():.3f}")
print(f"Feature scales (train std): {X_train.std(axis=0).round(2)}")

---

## Part 1 (15 points, coding task)

**Implement `StandardScaler`**: zero mean, unit variance.

- `fit(X)`: compute per-feature mean and std from training data.
- `transform(X)`: apply standardization using stored statistics.
- `fit_transform(X)`: convenience method.

**Critical**: fit on training data only, then transform both train and test using the *same* statistics.

*Reasoning is not required.*

In [ ]:
class StandardScaler:
    """
    Standardize features: zero mean, unit variance.
    Fit on training data, apply to any data.
    """
    def __init__(self):
        self.mean_ = None
        self.std_ = None

    def fit(self, X: np.ndarray) -> 'StandardScaler':
        """Compute mean and std per feature. X: (n, d)"""
        ### WRITE YOUR SOLUTION HERE ###

        pass

    def transform(self, X: np.ndarray) -> np.ndarray:
        """Apply standardization. X: (n, d) -> (n, d)"""
        ### WRITE YOUR SOLUTION HERE ###

        pass

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        return self.fit(X).transform(X)

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)                        # (400, 10)
X_test_s = scaler.transform(X_test)                              # (100, 10)

assert X_train_s.shape == X_train.shape
assert X_test_s.shape == X_test.shape
assert np.allclose(X_train_s.mean(axis=0), 0, atol=1e-10), "Train mean should be ~0"
assert np.allclose(X_train_s.std(axis=0), 1, atol=1e-10), "Train std should be ~1"
print(f"Train mean: {X_train_s.mean(axis=0).round(6)}")
print(f"Train std:  {X_train_s.std(axis=0).round(6)}")
print("Part 1 passed.")

""" END OF THIS PART """

---

## Part 2 (15 points, coding task)

**Implement polynomial feature expansion.**

For degree 2: given $X \in \mathbb{R}^{n \times d}$, append all pairwise products $x_i x_j$ for $i \leq j$ (including $x_i^2$). Return the augmented matrix.

Also implement a helper to add a bias column of ones.

*Reasoning is not required.*

In [ ]:
def add_polynomial_features(X: np.ndarray, degree: int = 2) -> np.ndarray:
    """
    Add pairwise interaction features (degree 2).
    Appends x_i * x_j for all i <= j.

    Args:
        X: (n, d) original features
        degree: only degree=2 is required

    Returns:
        X_poly: (n, d + d*(d+1)/2) augmented feature matrix
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass


def add_bias(X: np.ndarray) -> np.ndarray:
    """Prepend a column of ones. X: (n, d) -> (n, d+1)"""
    ### WRITE YOUR SOLUTION HERE ###

    pass

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
X_small = np.array([[1, 2], [3, 4]])
X_poly_small = add_polynomial_features(X_small)
# Original (2) + pairs: x1^2, x1*x2, x2^2 = 3 -> total 5
assert X_poly_small.shape[1] == 5, f"Expected 5 columns, got {X_poly_small.shape[1]}"
print(f"Poly features shape: {X_poly_small.shape}")
print(f"Row 0: {X_poly_small[0]}")

X_bias = add_bias(X_small)
assert X_bias.shape == (2, 3)
assert np.all(X_bias[:, 0] == 1)
print("Part 2 passed.")

""" END OF THIS PART """

---

### Model Comparison via Cross-Validation

To fairly compare models, use the **same cross-validation splits** and evaluate with consistent metrics. The best model is chosen based on CV performance, then evaluated once on the held-out test set.

---

## Part 3 (20 points, coding task)

**Implement a cross-validation framework and compare at least 3 models.**

1. Implement `cross_validate(X, y, train_fn, predict_fn, metric_fn, k=5)` that:
   - Splits data into `k` folds
   - For each fold: trains a model, predicts, evaluates
   - Returns `(mean_score, std_score, per_fold_scores)`

2. Compare at least 3 models (use your implementations from previous assignments or re-implement here):
   - Logistic regression (on standardized features with bias)
   - Logistic regression + polynomial features
   - kNN (with a reasonable $k$)

3. Print a table of results.

*Reasoning is not required.*

In [ ]:
def cross_validate(
    X: np.ndarray,
    y: np.ndarray,
    train_fn,        # callable(X_train, y_train) -> model
    predict_fn,      # callable(model, X_test) -> y_pred
    metric_fn,       # callable(y_true, y_pred) -> score
    k: int = 5,
) -> tuple:
    """
    k-fold cross-validation.

    Returns:
        mean_score: float
        std_score: float
        per_fold_scores: list of k floats
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass

In [ ]:
### WRITE YOUR SOLUTION HERE ###
# Re-implement or copy key functions: sigmoid, logistic_regression,
# knn_classify, pairwise_distances, etc.
#
# Define train_fn and predict_fn for each model.
# Run cross_validate on X_train_s (standardized) for each.
# Print results table:
#
# | Model                     | CV Accuracy (mean +/- std) |
# |---------------------------|----------------------------|
# | Logistic (basic)          |                            |
# | Logistic (poly features)  |                            |
# | kNN (k=?)                 |                            |

pass

""" END OF THIS PART """

---

## Part 4 (25 points, coding task)

**Final evaluation on the test set.**

1. Select the best model based on CV results.
2. Train it on the **full training set** (standardized).
3. Evaluate on the test set. Compute and report: **accuracy, precision, recall, F1**.
4. Plot two figures:
   - **ROC curve** with AUC annotation (if your model produces scores; otherwise plot confusion matrix).
   - **Confusion matrix** as a heatmap.

*Reasoning is not required.*

In [ ]:
### WRITE YOUR SOLUTION HERE ###
# Train best model on full training set
# Evaluate on test set
# Print: accuracy, precision, recall, F1
# Plot: ROC curve and/or confusion matrix

pass

""" END OF THIS PART """

---

## Part 5 (15 points, non-coding task)

**Results report.**

Write a brief report (5-8 sentences) addressing:

1. Which model performed best and why? Relate to the data generation process (nonlinear boundary, interaction terms, irrelevant features).
2. What was the impact of standardization? What would happen if you skipped it for kNN vs logistic regression?
3. Why did you choose the metrics you reported? Which metric is most appropriate for this dataset and why?
4. What are the limitations of your pipeline? Name one improvement you could make.

*Reasoning is required.*

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

---

## Part 6 (10 points, non-coding task)

**Pipeline design questions.**

1. Explain why you must fit the `StandardScaler` on training data only and then transform the test data with the *same* statistics. What goes wrong if you fit on the combined (train + test) data?

2. You used 5-fold CV for model selection and then evaluated on a held-out test set. Why is this two-level evaluation necessary? What bias would you introduce if you selected the model based on test-set performance?

*Reasoning is required.*

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """